# Занятие 3. MCP, A2A и агентные системы

**Цель занятия:** построить агента, который оркеструет модули из занятий 1–2 через стандартные протоколы,
и научиться его оценивать.

| Блок | Время | Содержание |
|---|---|---|
| 1 | 10 мин | Теория агентов: tool use → ReAct → планировщики → мультиагентность |
| 2 | 15 мин | MCP: зачем, архитектура, минимальный сервер + клиент |
| 3 | 12 мин | A2A: обмен задачами между агентами, сравнение с MCP |
| 4 | 15 мин | LangGraph: граф plan → act → observe с циклом |
| 5 | 8 мин | Structured output и retry на невалидном ответе |
| 6 | 10 мин | Оценка: RAGAS / кастомные метрики, task success rate, latency |
| 7 | 5 мин | Обзор экосистем |
| 8 | 5 мин | Q&A + ДЗ |

**Разделение протоколов:**
* **MCP (Model Context Protocol)** — агент ↔ инструмент. Стандартизирует, как модель находит и вызывает тулы,
  читает ресурсы и получает промпты.
* **A2A (Agent-to-Agent)** — агент ↔ агент. Описание возможностей агента (Agent Card), жизненный цикл задачи,
  обмен сообщениями и артефактами.

**Про железо.** GPU не требуется: MCP-сервер, граф и метрики считаются на CPU.

In [ ]:
import os, sys, json, time, asyncio, warnings, subprocess, textwrap
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Literal, Tuple

import numpy as np

warnings.filterwarnings("ignore")

IN_COLAB = "COLAB_GPU_TOPOLOGY" in os.environ or os.path.exists("/content")
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "mcp", "langgraph", "langchain-core", "pydantic", "ragas"])

ART = "/home/user/course/artifacts"
os.makedirs(ART, exist_ok=True)
SEED = 42
np.random.seed(SEED)

import mcp as _mcp
import pydantic
print(f"mcp {_mcp.__file__.split('site-packages/')[-1].split('/')[0]} | pydantic {pydantic.__version__}")
try:
    import langgraph, importlib.metadata as _md
    print("langgraph", _md.version("langgraph"))
except Exception as e:
    print("langgraph:", e)

mcp  | pydantic 2.14.2
langgraph 1.2.19


## Зачем это и где применяется

**Зачем.** Модель сама по себе не умеет ничего сделать во внешнем мире: ей нужны инструменты.
Пока вызовы инструментов написаны «в лоб» внутри одного скрипта, их нельзя переиспользовать
в другом проекте или передать другому агенту. MCP задаёт стандарт «агент ↔ инструмент»,
A2A — «агент ↔ агент», а граф состояний даёт управляемый цикл: повтор при неудаче и проверку результата.

**Где применяется.** Корпоративные ассистенты с доступом к базам и сервисам; робототехника
(один агент на зону, задачи передаются по A2A); RAG-системы, где ответ проверяется по источникам.

## Термины занятия (глоссарий)

| Термин | Смысл |
|---|---|
| **MCP** | Model Context Protocol: как модель находит и вызывает инструменты; сервер отдаёт tools / resources / prompts |
| **Tool** | инструмент: функция с именем, описанием и JSON-схемой аргументов |
| **Resource / Prompt** | данные, доступные по URI / шаблон промпта, который хранит сервер |
| **A2A** | Agent-to-Agent: обмен задачами между агентами (не путать с MCP: там агент ↔ инструмент) |
| **Agent Card** | карточка агента: имя, роль, навыки — по ней реестр выбирает исполнителя задачи |
| **Artifact** | результат работы, передаваемый вместе с задачей: диагноз, рекомендация, файл |
| **StateGraph** | граф, где узлы меняют общее состояние, а рёбра задают переходы |
| **Условное ребро** | переход, который выбирает функция-маршрутизатор: «повторить» или «завершить» |
| **Structured output** | ответ строго по схеме (pydantic / JSON Schema) — такой ответ можно провалидировать |
| **Faithfulness** | следует ли ответ из полученных наблюдений, нет ли выдумки |
| **Task success rate** | доля сценариев, доведённых до правильного результата |
| **p95 latency** | время, хуже которого только 5% запросов; важнее среднего, потому что описывает хвост |


### Архитектура занятия и выбор инструментов

```
пользователь ──▶ граф LangGraph: planner → executor → evaluator → finalize
                                      │                  │
                                      ▼                  └── retry (лимит попыток)
                               MCP-инструменты
                         (classify / transcribe / decide)

агент-диагност ══A2A══▶ агент-рекомендатор      (задача + артефакты)
```

| Решение | Пояснение |
|---|---|
| MCP поверх stdio | инструмент работает в отдельном процессе: тот же протокол, если модель или БД на другой машине |
| `MCPServer` (mcp 2.x) | в mcp 1.x класс назывался `FastMCP` и был переименован — отсюда такой импорт |
| LangGraph | нужен цикл (evaluator → executor) и явное состояние; на «просто функциях» это быстро превращается в набор флагов |
| pydantic | ответ проверяется по схеме до попадания в систему; невалидный уходит в retry с логом |
| Детерминированные метрики | RAGAS требует LLM-судью и ключ; без ключа считаем метрики по наблюдениям и честно указываем это в отчёте |
| `asyncio` в отдельном потоке | ядро IPython уже запустило event loop, поэтому `asyncio.run()` в ноутбуке падает |


## 1. Архитектура агента

```
             ┌──────────── Planner (LLM) ────────────┐
             │  решает, что делать дальше            │
             └───────────────┬───────────────────────┘
                             │ plan
             ┌───────────────▼───────────────────────┐
             │  Executor: вызов инструментов (MCP)   │──── tools: search, calc, DB, ML-модели
             └───────────────┬───────────────────────┘
                             │ observation
             ┌───────────────▼───────────────────────┐
             │  Evaluator: достаточно ли?  ──нет─────┼──> обратно в Planner (цикл!)
             └───────────────┬───────────────────────┘
                            да
                             ▼
                          ответ
```

Ключевые компоненты: **инструменты** (схема вызова), **память** (состояние диалога/задачи),
**планировщик**, **исполнитель**, **оценщик**.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame([
    dict(паттерн="Tool use",      идея="модель вызывает функции по JSON-схеме",          когда="1 шаг, одна задача"),
    dict(паттерн="ReAct",         идея="чередование Reasoning и Acting",                когда="нужно несколько шагов"),
    dict(паттерн="Plan-Execute",  идея="сначала план целиком, потом исполнение",         когда="длинные задачи"),
    dict(паттерн="Reflection",    идея="оценщик проверяет результат и даёт retry",       когда="важна надёжность"),
    dict(паттерн="Multi-agent",   идея="несколько агентов с ролями обмениваются A2A",    когда="задача шире одной роли"),
]))
print("Главная ошибка новичков: агент без оценки и без ограничения числа шагов.")
print("Без evaluator вы не узнаете, что агент «зациклился»; без лимита — он будет жечь токены вечно.")

,паттерн,идея,когда
0,Tool use,модель вызывает функции по JSON-схеме,"1 шаг, одна задача"
1,ReAct,чередование Reasoning и Acting,нужно несколько шагов
2,Plan-Execute,"сначала план целиком, потом исполнение",длинные задачи
3,Reflection,оценщик проверяет результат и даёт retry,важна надёжность
4,Multi-agent,несколько агентов с ролями обмениваются A2A,задача шире одной роли


Главная ошибка новичков: агент без оценки и без ограничения числа шагов.
Без evaluator вы не узнаете, что агент «зациклился»; без лимита — он будет жечь токены вечно.


## 2. MCP: Model Context Protocol

Архитектура: **host** (приложение) → **client** → **server** (инструменты).
Сервер отдаёт три вида сущностей:

* **tools** — вызываемые функции с JSON-схемой;
* **resources** — данные, которые можно прочитать (файлы, конфиги, справочники);
* **prompts** — шаблоны промптов.

Ниже — реальный MCP-сервер на Python, который экспортирует инструменты нашего проекта
(`classify_image`, `transcribe_audio`, `decide`). Внутри стоят заглушки-стабы: они реализуют
те же методы, что у модулей из ДЗ-1/ДЗ-2, поэтому замена на реальные модели — одна строка.

In [ ]:
mcp_server_code = r'''
"""MCP-сервер проекта: инструменты восприятия и «мозга».

Запуск: python3 mcp_server.py  (общение по stdio)
API: mcp 2.x — класс MCPServer (в 1.x он назывался FastMCP).
"""
import json
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("project-tools")

# --- Заглушки модулей (в проекте сюда подключаются MultimodalPerception и LLMBrain из ДЗ-1/ДЗ-2)
CLASSES = ["circle", "square", "triangle", "cross"]
COMMANDS = ["stop", "forward", "back", "left", "right", "home"]
FAULT_CAUSE = {"pump": "bearing wear", "compressor": "low oil level", "conveyor": "low belt tension",
               "gearbox": "lack of lubrication", "pipeline": "leak in the loop", "fan": "clogged filter"}

def _hash(text: str) -> int:
    import hashlib
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

@mcp.tool()
def classify_image(image_uri: str) -> dict:
    """Классифицировать изображение (заглушка детерминированного восприятия).

    Args:
        image_uri: путь или идентификатор изображения
    Returns:
        словарь с меткой класса и уверенностью
    """
    label = CLASSES[_hash(image_uri) % len(CLASSES)]
    return {"label": label, "confidence": round(0.55 + (_hash(image_uri + "c") % 40) / 100, 2),
            "source": image_uri}

@mcp.tool()
def transcribe_audio(audio_uri: str) -> dict:
    """Распознать голосовую команду (заглушка ASR).

    Args:
        audio_uri: путь или идентификатор аудиозаписи
    Returns:
        словарь с текстом команды
    """
    return {"command": COMMANDS[_hash(audio_uri) % len(COMMANDS)], "source": audio_uri}

@mcp.tool()
def decide(object_label: str, command: str, confidence: float = 1.0) -> dict:
    """Принять решение на основе восприятия (заглушка мозга).

    Args:
        object_label: распознанный класс объекта
        command: распознанная голосовая команда
        confidence: уверенность восприятия
    Returns:
        действие агента и пояснение
    """
    actions = {"stop": "остановиться", "forward": "двигаться вперёд", "back": "двигаться назад",
               "left": "повернуть влево", "right": "повернуть вправо", "home": "вернуться на базу"}
    action = actions.get(command, "уточнить команду")
    if confidence < 0.6:
        return {"action": "запросить подтверждение оператора",
                "reason": f"уверенность восприятия {confidence} ниже порога 0.6"}
    return {"action": f"{action}: объект '{object_label}'", "reason": f"команда '{command}' распознана"}

@mcp.resource("config://robot/safety")
def safety_config() -> str:
    """Правила безопасности робота (регламент)."""
    return json.dumps({"min_confidence": 0.6, "max_speed": 0.8, "stop_words": ["stop", "аварийно"]},
                      ensure_ascii=False)

@mcp.prompt()
def operator_report(object_label: str, action: str) -> str:
    """Шаблон отчёта для оператора."""
    return f"Объект: {object_label}. Выполнено действие: {action}. Подтвердите корректность."

if __name__ == "__main__":
    # транспорт stdio: клиент сам запускает этот файл отдельным процессом и общается через stdin/stdout
    mcp.run(transport="stdio")
'''

server_path = f"{ART}/mcp_server.py"
open(server_path, "w").write(mcp_server_code)
print(f"Сервер записан: {server_path} ({len(mcp_server_code.splitlines())} строк)")

Сервер записан: /home/user/course/artifacts/mcp_server.py (78 строк)


### 2.1 Клиент MCP: подключаемся и вызываем инструменты

In [ ]:
import asyncio, subprocess
from concurrent.futures import ThreadPoolExecutor
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def run_async(coro):
    """Запустить корутину из ноутбука в отдельном потоке со своим event loop."""
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()

# errlog — настоящий файл, иначе subprocess.Popen дёргает fileno() у ipykernel.iostream.OutStream
_ERRLOG = open(os.path.join(ART, "mcp_server_stderr.log"), "w")

async def mcp_demo():
    params = StdioServerParameters(
        command=sys.executable,
        args=[server_path],
    )
    # errlog передаём в stdio_client: так anyio не будет пытаться взять fileno() у sys.stderr ноутбука
    async with stdio_client(params, errlog=_ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])

            r1 = await session.call_tool("classify_image", {"image_uri": "cam1/frame_001.png"})
            print("classify_image ->", r1.content[0].text)

            r2 = await session.call_tool("transcribe_audio", {"audio_uri": "mic/utterance_01.wav"})
            print("transcribe_audio ->", r2.content[0].text)

            cmd = json.loads(r2.content[0].text)["command"]
            obj = json.loads(r1.content[0].text)["label"]
            conf = json.loads(r1.content[0].text)["confidence"]
            r3 = await session.call_tool("decide", {"object_label": obj, "command": cmd, "confidence": conf})
            print("decide ->", r3.content[0].text)

            resources = await session.list_resources()
            print("resources:", [str(r.uri) for r in resources.resources])
            rr = await session.read_resource("config://robot/safety")
            print("safety config ->", rr.contents[0].text)

            prompts = await session.list_prompts()
            print("prompts:", [p.name for p in prompts.prompts])
            return r3.content[0].text

decision_json = run_async(mcp_demo())

tools: ['classify_image', 'transcribe_audio', 'decide']
classify_image -> {
  "label": "cross",
  "confidence": 0.68,
  "source": "cam1/frame_001.png"
}
transcribe_audio -> {
  "command": "forward",
  "source": "mic/utterance_01.wav"
}
decide -> {
  "action": "двигаться вперёд: объект 'cross'",
  "reason": "команда 'forward' распознана"
}
resources: ['config://robot/safety']
safety config -> {"min_confidence": 0.6, "max_speed": 0.8, "stop_words": ["stop", "аварийно"]}
prompts: ['operator_report']


### 2.2 Обёртка инструментов в виде обычных функций Python

Агенту неудобно работать с `async`-сессией напрямую — завернём MCP-вызовы в синхронные функции.
Это же место — точка подмены: здесь стоят стабы, в проекте — `MultimodalPerception` и `LLMBrain`.

In [ ]:
import os, subprocess, json

# один общий файл для stderr сервера — создаём заранее, чтобы fileno() был настоящим
_ERRLOG_PATH = os.path.join(ART, "mcp_server_stderr.log")
_ERRLOG = open(_ERRLOG_PATH, "a", buffering=1)   # line-buffered, чтобы видеть логи сразу

def call_mcp(tool: str, **kwargs) -> Dict:
    """Синхронная обёртка над MCP-инструментом (каждый вызов — новая stdio-сессия)."""
    async def _run():
        params = StdioServerParameters(command=sys.executable, args=[server_path])
        # errlog=... обязателен: иначе anyio/subprocess дёргает fileno() у ipykernel.iostream.OutStream
        async with stdio_client(params, errlog=_ERRLOG) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                res = await session.call_tool(tool, kwargs)
                return json.loads(res.content[0].text)
    return run_async(_run())

TOOLS = {
    "classify_image": lambda image_uri: call_mcp("classify_image", image_uri=image_uri),
    "transcribe_audio": lambda audio_uri: call_mcp("transcribe_audio", audio_uri=audio_uri),
    "decide": lambda object_label, command, confidence=1.0: call_mcp(
        "decide", object_label=object_label, command=command, confidence=confidence),
}
print("Пример вызова обёртки:", TOOLS["classify_image"]("cam1/frame_002.png"))

Пример вызова обёртки: {'label': 'triangle', 'confidence': 0.77, 'source': 'cam1/frame_002.png'}


## 3. A2A: Agent-to-Agent

MCP — про «агент вызывает инструмент». A2A — про «агент поручает задачу другому агенту»:

| | MCP | A2A |
|---|---|---|
| Кто взаимодействует | агент ↔ инструмент | агент ↔ агент |
| Описание возможностей | JSON-схема инструмента | Agent Card (роль, навыки, форматы) |
| Единица работы | вызов функции | задача (task) с жизненным циклом |
| Результат | результат вызова | артефакты + статусы + сообщения |

Ниже — минимальная, но рабочая реализация A2A-подобного обмена: `Task` → `Message` → `Artifact`,
два агента («диагност» и «рекомендатор») обмениваются задачей и возвращают артефакт.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from enum import Enum
from uuid import uuid4

class TaskState(str, Enum):
    submitted = "submitted"
    working = "working"
    completed = "completed"
    failed = "failed"

class Message(BaseModel):
    role: Literal["user", "agent"]
    parts: List[Dict[str, Any]] = Field(default_factory=list)     # части: text / data
    def text(self) -> str:
        return " ".join(p.get("text", "") for p in self.parts)

class Task(BaseModel):
    id: str = Field(default_factory=lambda: uuid4().hex[:8])
    state: TaskState = TaskState.submitted
    messages: List[Message] = Field(default_factory=list)
    artifacts: List[Dict[str, Any]] = Field(default_factory=list)

class AgentCard(BaseModel):
    """Описание агента: что он умеет (аналог Agent Card из A2A)."""
    name: str
    description: str
    skills: List[str]

FAULT_CAUSE = {"pump": "bearing wear", "compressor": "low oil level", "conveyor": "low belt tension",
               "gearbox": "lack of lubrication", "pipeline": "leak in the loop", "fan": "clogged filter"}
RECOMMENDATIONS = {"bearing wear": "заменить подшипник", "low oil level": "долить масло",
                   "low belt tension": "натянуть ленту", "lack of lubrication": "проверить смазку",
                   "leak in the loop": "устранить утечку", "clogged filter": "заменить фильтр"}

def diagnose(task: Task) -> Task:
    """Агент-диагност: по тексту задачи формирует гипотезу неисправности."""
    task.state = TaskState.working
    text = task.messages[-1].text()
    equipment = next((w for w in text.lower().split() if w in FAULT_CAUSE), "pump")
    cause = FAULT_CAUSE.get(equipment, "unknown")
    task.artifacts.append({"kind": "diagnosis", "equipment": equipment, "hypothesis": cause,
                           "confidence": 0.72})
    task.messages.append(Message(role="agent", parts=[{"text": f"Гипотеза: {cause} ({equipment})"}]))
    task.state = TaskState.completed
    return task

def recommend(task: Task) -> Task:
    """Агент-рекомендатор: по артефакту-диагнозу подбирает действие и приоритет."""
    task.state = TaskState.working
    diagnosis = next((a for a in task.artifacts if a.get("kind") == "diagnosis"), None)
    if diagnosis is None:
        task.state = TaskState.failed
        task.messages.append(Message(role="agent", parts=[{"text": "нет диагноза в артефактах"}]))
        return task
    cause = diagnosis["hypothesis"]
    task.artifacts.append({"kind": "recommendation", "action": RECOMMENDATIONS.get(cause, "вызвать инженера"),
                           "priority": "high" if diagnosis["confidence"] > 0.7 else "normal"})
    task.messages.append(Message(role="agent", parts=[{"text": f"Действие: {RECOMMENDATIONS.get(cause)}"}]))
    task.state = TaskState.completed
    return task

# Реестр агентов: карточка + функция-обработчик. Выбор по навыку, без if/elif по имени агента.
AGENTS = {
    "diagnose": (AgentCard(name="diagnostic", description="ставит диагноз по симптомам",
                           skills=["diagnose"]), diagnose),
    "recommend": (AgentCard(name="advisor", description="даёт рекомендацию по диагнозу",
                            skills=["recommend"]), recommend),
}

def delegate(skill: str, text: str, artifacts=None) -> Task:
    """Передать задачу агенту по навыку (мини-версия A2A)."""
    task = Task(messages=[Message(role="user", parts=[{"text": text}])],
                artifacts=list(artifacts or []))     # артефакты — способ передать контекст
    if skill not in AGENTS:
        task.state = TaskState.failed
        return task
    return AGENTS[skill][1](task)

print("Карточки агентов:", [card.name for card, _ in AGENTS.values()])

# round-trip: диагност ставит гипотезу, рекомендатор получает задачу вместе с артефактом диагноза
task = delegate("diagnose", "На compressor наблюдается перегрев, нужно понять причину")
print("после диагноста:", task.state.value, json.dumps(task.artifacts, ensure_ascii=False))

task2 = delegate("recommend", "нужна рекомендация по диагнозу", artifacts=task.artifacts)
print("Состояние задачи:", task2.state.value)
print("Артефакты:", json.dumps(task2.artifacts, ensure_ascii=False))
print("Сообщения:", [m.text() for m in task2.messages])

Карточки агентов: ['diagnostic', 'advisor']
после диагноста: completed [{"kind": "diagnosis", "equipment": "compressor", "hypothesis": "low oil level", "confidence": 0.72}]
Состояние задачи: completed
Артефакты: [{"kind": "diagnosis", "equipment": "compressor", "hypothesis": "low oil level", "confidence": 0.72}, {"kind": "recommendation", "action": "долить масло", "priority": "high"}]
Сообщения: ['нужна рекомендация по диагнозу', 'Действие: долить масло']


## 4. LangGraph: граф plan → act → observe с циклом

Граф состояний: узлы — агенты/функции, рёбра — переходы, условные рёбра — ветвление.
Цикл нужен, чтобы evaluator мог отправить задачу на повтор (retry), если качество не устраивает.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    question: str
    plan: str
    observation: str
    answer: str
    tries: int
    ok: bool
    log: List[str]

MAX_TRIES = 3          # лимит попыток: без него граф может крутиться бесконечно

def planner(state: AgentState) -> Dict:
    """Определяет, какие инструменты нужны (в реальности — вызов LLM)."""
    q = state["question"].lower()
    need = "image" if "изображени" in q or "камер" in q or "image" in q else "audio"
    return {"plan": need, "tries": state.get("tries", 0),
            "log": state.get("log", []) + [f"plan: нужен инструмент '{need}'"]}

def executor(state: AgentState) -> Dict:
    """Исполняет шаг плана через MCP-инструменты."""
    plan, log = state["plan"], state.get("log", [])
    if plan == "image":
        r = TOOLS["classify_image"](state["question"][:32])
        obs = f"объект={r['label']} (уверенность {r['confidence']})"
    else:
        r = TOOLS["transcribe_audio"](state["question"][:32])
        obs = f"команда={r['command']}"
    return {"observation": obs, "log": log + [f"act: {obs}"]}

def evaluator(state: AgentState) -> Dict:
    """Проверяет результат и решает: хватит или повторить."""
    log = state.get("log", [])
    tries = state.get("tries", 0) + 1
    obs = state.get("observation", "")
    ok = ("уверенность 0." in obs and float(obs.split("уверенность ")[1].rstrip(")")) >= 0.6) or \
         ("команда=" in obs and obs.split("команда=")[1].strip() != "")
    answer = f"Ответ: {obs}" if ok else ""
    return {"ok": ok, "tries": tries, "answer": answer,
            "log": log + [f"evaluate: ok={ok} (попытка {tries})"]}

def finalize(state: AgentState) -> Dict:
    """Формирует финальный ответ. Важно: агент обязан что-то вернуть даже при неудаче —
    пустой ответ («молчание») — худший вариант для оператора."""
    log = state.get("log", [])
    if state["ok"]:
        return {"log": log + ["finalize: ответ подтверждён"]}
    return {"answer": f"Не удалось подтвердить результат: {state.get('observation', '')} (уверенность ниже порога)",
            "log": log + ["finalize: ответ с пометкой «не подтверждено»"]}

def route_after_eval(state: AgentState) -> Literal["executor", "finalize"]:
    """Условный переход: повтор, если не прошли проверку и остались попытки."""
    if state["ok"] or state["tries"] >= MAX_TRIES:
        return "finalize"
    return "executor"

# Узел графа — функция, которая получает состояние и возвращает ИЗМЕНЕНИЯ состояния
# (не всё состояние целиком: LangGraph сам склеивает результат с текущим состоянием).
graph = StateGraph(AgentState)
graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("evaluator", evaluator)
graph.add_node("finalize", finalize)
graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "evaluator")
graph.add_conditional_edges("evaluator", route_after_eval, {"executor": "executor", "finalize": "finalize"})
graph.add_edge("finalize", END)
app = graph.compile()          # compile() превращает описание графа в исполняемое приложение

# invoke прогоняет граф от START до END и возвращает итоговое состояние (включая log всех узлов)
result = app.invoke({"question": "Что видно на изображении с камеры 1?", "plan": "", "observation": "",
                     "answer": "", "tries": 0, "ok": False, "log": []})
print("\n".join(result["log"]))
print("\nИтог:", result["answer"], "| попыток:", result["tries"])
try:
    print("\nСхема графа (mermaid):")
    print(app.get_graph().draw_mermaid())
except Exception as e:
    print("mermaid недоступен:", e)

plan: нужен инструмент 'image'
act: объект=square (уверенность 0.58)
evaluate: ok=False (попытка 1)
act: объект=square (уверенность 0.58)
evaluate: ok=False (попытка 2)
act: объект=square (уверенность 0.58)
evaluate: ok=False (попытка 3)
finalize: ответ с пометкой «не подтверждено»

Итог: Не удалось подтвердить результат: объект=square (уверенность 0.58) (уверенность ниже порога) | попыток: 3

Схема графа (mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	executor(executor)
	evaluator(evaluator)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	evaluator -.-> executor;
	evaluator -.-> finalize;
	executor --> evaluator;
	planner --> executor;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. Structured output и retry

Агент обязан возвращать **структурированный** результат: его проще валидировать, логировать
и передавать дальше (например, в MCP-тул робота). Ниже — pydantic-схема и цикл с повтором:
модель (здесь — имитация) иногда выдаёт невалидный JSON или значение вне диапазона.

In [ ]:
class ActionPlan(BaseModel):
    action: str = Field(..., description="одно из: move/stop/inspect/ask_operator")
    target: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0.0, le=1.0)
    comment: str = ""

VALID_ACTIONS = {"move", "stop", "inspect", "ask_operator"}

def fake_llm(attempt: int) -> str:
    """Имитация LLM: первые две попытки — невалидный ответ, третья — нормальный."""
    bad = [
        '{"action": "fly", "target": "pump", "confidence": 1.7}',          # действие вне списка и confidence>1
        '{"action": "move", "target": "", "confidence": 0.9}',             # пустая цель
    ]
    good = '{"action": "move", "target": "pump", "confidence": 0.86, "comment": "к насосу"}'
    return bad[attempt] if attempt < len(bad) else good

def parse_with_retry(generate_fn, max_attempts: int = 3) -> Tuple[Optional[ActionPlan], List[Dict]]:
    # первая попытка модели часто невалидна, поэтому генерация обёрнута в retry с логом
    log = []
    for attempt in range(max_attempts):
        raw = generate_fn(attempt)
        try:
            plan = ActionPlan.model_validate_json(raw)
            if plan.action not in VALID_ACTIONS:
                raise ValidationError.from_exception_data(
                    "ActionPlan", [{"type": "value_error",
                                    "loc": ("action",),
                                    "msg": f"action '{plan.action}' не в {sorted(VALID_ACTIONS)}",
                                    "input": plan.action}])
            log.append({"attempt": attempt + 1, "status": "ok", "raw": raw})
            return plan, log
        except ValidationError as e:
            log.append({"attempt": attempt + 1, "status": "invalid",
                        "raw": raw, "error": str(e).splitlines()[1].strip()[:80] if len(str(e).splitlines()) > 1 else str(e)[:80]})
    return None, log

plan, retry_log = parse_with_retry(fake_llm)
for row in retry_log:
    print(row)
print("\nИтоговый валидный план:", plan)

{'attempt': 1, 'status': 'invalid', 'raw': '{"action": "fly", "target": "pump", "confidence": 1.7}', 'error': 'confidence'}
{'attempt': 2, 'status': 'invalid', 'raw': '{"action": "move", "target": "", "confidence": 0.9}', 'error': 'target'}
{'attempt': 3, 'status': 'ok', 'raw': '{"action": "move", "target": "pump", "confidence": 0.86, "comment": "к насосу"}'}

Итоговый валидный план: action='move' target='pump' confidence=0.86 comment='к насосу'


## 6. Оценка агента

Три уровня метрик:

1. **Пошаговые** — правильность вызова инструментов, доля валидных structured output.
2. **Качество ответа** — faithfulness («ответ следует из наблюдений») и relevancy.
   RAGAS считает их через LLM-судью; без API используем детерминированный приближённый вариант.
3. **Системные** — task success rate и end-to-end latency.

In [ ]:
SCENARIOS = [
    {"id": 1, "question": "Что видно на изображении с камеры 1?", "expect_modality": "image", "expect_keys": ["объект"]},
    {"id": 2, "question": "Распознай команду оператора с микрофона", "expect_modality": "audio", "expect_keys": ["команда"]},
    {"id": 3, "question": "Определи объект по image из архива", "expect_modality": "image", "expect_keys": ["объект"]},
    {"id": 4, "question": "Что сказали в audio-сообщении?", "expect_modality": "audio", "expect_keys": ["команда"]},
    {"id": 5, "question": "Классифицируй изображение с камеры 2", "expect_modality": "image", "expect_keys": ["объект"]},
]

def faithfulness(answer: str, observation: str) -> float:
    """Доля слов ответа, которые встречаются в наблюдении (детерминированная аппроксимация)."""
    a = set(answer.lower().replace("=", " ").replace("(", " ").replace(")", " ").split())
    o = set(observation.lower().replace("=", " ").replace("(", " ").replace(")", " ").split())
    return round(len(a & o) / max(len(a), 1), 3)

def relevancy(answer: str, expected_keys: List[str]) -> float:
    """Релевантность: доля ожидаемых ключей в ответе минус штраф за многословие.
    (Словесное пересечение вопрос↔ответ здесь не работает: вопрос на русском, ответ — структурированный.)"""
    a = answer.lower()
    base = sum(1 for k in expected_keys if k in a) / max(len(expected_keys), 1)
    penalty = max(0, len(answer.split()) - 20) * 0.02
    return round(max(0.0, base - penalty), 3)

# прогоняем все сценарии через граф и замеряем end-to-end latency (время всего цикла, а не одного узла)
rows, latencies = [], []
for sc in SCENARIOS:
    t0 = time.time()
    res = app.invoke({"question": sc["question"], "plan": "", "observation": "",
                      "answer": "", "tries": 0, "ok": False, "log": []})
    dt = time.time() - t0
    latencies.append(dt)
    correct_modality = res["plan"] == sc["expect_modality"]
    has_keys = all(k in res["answer"] for k in sc["expect_keys"])
    rows.append(dict(scenario=sc["id"], модальность=res["plan"], верная_модальность=correct_modality,
                     есть_ключи=has_keys, успех=bool(correct_modality and has_keys and res["ok"]),
                     faithfulness=faithfulness(res["answer"], res["observation"]),
                     relevancy=relevancy(res["answer"], sc["expect_keys"]),
                     latency_с=round(dt, 3)))

df_eval = pd.DataFrame(rows)
display(df_eval)
# p95 важнее среднего: он показывает хвост — самые медленные 5% прогонов, которые и портят опыт
print(f"Task success rate: {df_eval['успех'].mean():.2f} "
      f"| средняя latency: {np.mean(latencies):.3f} с "
      f"| p95 latency: {np.percentile(latencies, 95):.3f} с")

,scenario,модальность,верная_модальность,есть_ключи,успех,faithfulness,relevancy,latency_с
0,1,image,True,True,False,0.400,1.0,5.268
1,2,audio,True,True,True,0.667,1.0,1.294
2,3,image,True,True,True,0.800,1.0,1.301
3,4,audio,True,True,True,0.667,1.0,1.293
4,5,image,True,True,True,0.800,1.0,1.310


Task success rate: 0.80 | средняя latency: 2.093 с | p95 latency: 4.476 с


### 6.1 RAGAS (если доступен)

RAGAS требует LLM-судью и API-ключ.

In [ ]:
def ragas_metrics_if_available(rows_df, questions) -> Optional[Any]:
    """RAGAS считает faithfulness/answer_relevancy через LLM-судью: нужен установленный пакет и ключ API.
    Без него возвращаем None и используем детерминированные метрики (так же честно, если указать это в отчёте)."""
    try:
        from ragas import evaluate as ragas_evaluate
        from ragas.metrics import faithfulness as ragas_faith, answer_relevancy
        from datasets import Dataset as HFDataset
        ds = HFDataset.from_list([{"question": q, "contexts": [str(r["модальность"])],
                                   "answer": str(r["faithfulness"])}
                                  for (_, r), q in zip(rows_df.iterrows(), questions)])
        return ragas_evaluate(ds, metrics=[ragas_faith, answer_relevancy])
    except Exception as e:
        print(f"RAGAS недоступен ({type(e).__name__}) — считаем детерминированные метрики")
        return None

ragas_result = ragas_metrics_if_available(df_eval, [sc["question"] for sc in SCENARIOS])
ragas_report = "RAGAS: недоступен (используем детерминированные метрики)" if ragas_result is None else "RAGAS: метрики посчитаны"
print(ragas_report)

RAGAS недоступен (OpenAIError) — считаем детерминированные метрики
RAGAS: недоступен (используем детерминированные метрики)


## 7. Обзор экосистем: когда что брать

| Фреймворк | Сила | Слабость | Когда брать |
|---|---|---|---|
| **LangGraph** | явный граф состояний, циклы, контроль | больше кода | сложная логика, нужны циклы и ветвления |
| **smolagents** | код-как-действие, очень короткие агенты | меньше контроля | прототипы, research-агенты |
| **LangChain** | огромная экосистема интеграций | абстракции могут мешать | много готовых коннекторов |
| **MS Agent Framework** | интеграция с Azure/enterprise | вендорлок | корпоративный стек Microsoft |

In [ ]:
try:
    from langchain_core.tools import tool as lc_tool

    @lc_tool
    def multiply(a: int, b: int) -> int:
        """Умножить два целых числа."""
        return a * b

    print("LangChain tool:", multiply.name, "->", multiply.invoke({"a": 6, "b": 7}))
except Exception as e:
    print("langchain-core недоступен:", type(e).__name__, e)

try:
    import smolagents
    print("smolagents доступен:", smolagents.__version__)
except Exception as e:
    print("smolagents не установлен (pip install smolagents) — в ДЗ-3 сравните его с LangGraph")

LangChain tool: multiply -> 42
smolagents не установлен (pip install smolagents) — в ДЗ-3 сравните его с LangGraph


## 8. Упражнения для самостоятельной работы (в ноутбуке)

1. Добавьте в MCP-сервер четвёртый инструмент `lookup_manual(section)` и вызовите его из графа.
2. Сделайте `MAX_TRIES = 1` и измерьте, как упадут task success rate и latency.
3. Реализуйте в A2A состояние `input-required`: агент просит уточнение и ждёт ответ.
4. Замените `faithfulness`/`relevancy` на LLM-as-a-judge и сравните корреляцию с детерминированными метриками.
5. Добавьте в граф узел-память, который копит наблюдения в состоянии между попытками.

In [ ]:
artifacts = {
    "mcp_tools": ["classify_image", "transcribe_audio", "decide"],
    "a2a_agents": [card.model_dump() for card, _ in AGENTS.values()],
    "graph_nodes": ["planner", "executor", "evaluator"],
    "eval": df_eval.to_dict("records"),
    "task_success_rate": round(float(df_eval["успех"].mean()), 3),
    "mean_latency_s": round(float(np.mean(latencies)), 3),
    "ragas": ragas_report,
}
json.dump(artifacts, open(f"{ART}/hw3_metrics.json", "w"), ensure_ascii=False, indent=1)
print("Сохранено:", f"{ART}/hw3_metrics.json")
print("Task success rate:", artifacts["task_success_rate"], "| MCP-инструменты:", artifacts["mcp_tools"])

Сохранено: /home/user/course/artifacts/hw3_metrics.json
Task success rate: 0.8 | MCP-инструменты: ['classify_image', 'transcribe_audio', 'decide']


---
# Домашнее задание 3. Агентная система и её оценка

**Цель:** построить агента, который оркеструет модули из ДЗ-1 и ДЗ-2, и научиться его оценивать.

### Что сделать

1. **Инструменты:** обернуть `MultimodalPerception` и `LLMBrain` как MCP-инструменты
   (минимум три тула: `classify_image`, `transcribe_audio`, `decide`).
2. **Агент-оркестратор** на LangGraph: `planner → executor → evaluator → ответ`,
   с условным переходом (не прошёл проверку — retry) и лимитом попыток.
3. **A2A-сценарий:** два агента (например, «диагност» и «рекомендатор») обмениваются задачей;
   хотя бы один полный round-trip с артефактами.
4. **Structured output:** все ответы валидируются pydantic-схемой; при невалидном ответе — retry с логом.
5. **Оценка:**
   * faithfulness / relevancy на ≥5 примерах (детерминированные метрики из раздела 6 или RAGAS при наличии ключа);
   * task success rate на 5 сценариях (вручную размеченный «успех/не успех»);
   * end-to-end latency (среднее и p95).
6. **Отчёт:** схема графа, таблица метрик, 2–3 вывода, что идёт в проект.

### Формат сдачи

* `hw3_<группа>.ipynb` (исполненный) + `hw3_report.md` + код MCP-сервера отдельным файлом.

### Технические условия

* GPU не требуется: MCP-сервер, LangGraph и метрики считаются на CPU.
* RAGAS нужен только при наличии ключа LLM-судьи; без ключа используйте детерминированные
  метрики из раздела 6 и укажите это в отчёте.

### Критерии оценки (максимум 10 баллов)

| Критерий | Баллы | Что проверяется |
|---|---|---|
| MCP-сервер | 3 | Реально запускается, ≥3 инструмента, клиент вызывает их, есть resource/prompt |
| Граф агента | 3 | LangGraph: planner/executor/evaluator, есть цикл, условный переход, лимит попыток |
| A2A | 1 | Два агента, round-trip, артефакты, описаны роли (Agent Card) |
| Structured output | 1 | pydantic-схема + retry-лог на невалидных ответах |
| Оценка | 2 | Метрики качества + task success rate + latency, указано количество примеров |
